In [ ]:
!pip install pytorch-ignite

In [ ]:
import torch
import torch.nn as nn
import torch.optim
from torch.utils.data import Dataset, DataLoader, random_split
from torch.nn.utils.rnn import pad_sequence
import math
import random
import string
import uuid

from ignite.engine import Engine, Events
from ignite.handlers import ModelCheckpoint, EarlyStopping, ProgressBar
from ignite.metrics import Loss, RunningAverage


In [ ]:
config = {
    "d_model": 128, # dimension
    "nhead": 4, # Number of attention heads
    "num_layers": 3,
    "dim_feedforward": 256,
    "dropout": 0.1,
    "max_seq_len": 50,
    "batch_size": 64,
    "lr": 1e-3,
    "epochs": 60,
    "train_samples": 50000,
    "val_samples": 5000,
    "test_samples": 5000,
    "min_str_len": 3,
    "max_str_len": 10,
    "seed": 42,
}


In [ ]:
random.seed(config["seed"])
torch.manual_seed(config["seed"])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(config["seed"])

In [ ]:
token_mapping = {"<PAD>": 0, "<SEP>": 1, "<EOS>": 2}
curr_val = 3

for char in string.ascii_lowercase:
    token_mapping[char] = curr_val
    curr_val += 1

In [ ]:
def encode(input_str: str) -> list[int]:
    return_lst = []
    for curr in input_str:
        return_lst.append(token_mapping[curr])

    return return_lst


In [ ]:
reverse_token_mapping = {}

for key in token_mapping:
    val = token_mapping[key]
    reverse_token_mapping[val] = key

In [ ]:
def decode(input_lst: list[int]) -> str:
    return_str = ""
    for i in input_lst:
        if i < 3:
            continue
        return_str += reverse_token_mapping[i]
    return return_str

In [ ]:
vocab_size = len(token_mapping)

In [ ]:
class StringReversalDataset(Dataset):
    def __init__(self, min_len, max_len, num_samples):
        self.all_samples = []

        for _ in range(num_samples):

            length = random.randint(min_len, max_len)
            input_str = "".join(random.choices(string.ascii_lowercase, k=length))
            output_str = input_str[::-1]

            datapoint = {
                "id": uuid.uuid4().hex,
                "input": input_str,
                "output": output_str
            }

            self.all_samples.append(datapoint)

    def __len__(self):
        return len(self.all_samples)

    def __getitem__(self, idx):
        datapoint = self.all_samples[idx]

        input_ids = encode(datapoint["input"])
        output_ids = encode(datapoint["output"])

        full_sequence = input_ids + [token_mapping["<SEP>"]] + output_ids + [token_mapping["<EOS>"]]

        loss_mask = [0] * len(full_sequence)
        seg_start = len(input_ids) + 1
        seg_end = len(input_ids) + 1 + len(output_ids) + 1

        for i in range(seg_start, seg_end):
            loss_mask[i] = 1

        return {
            "input_ids": torch.tensor(full_sequence, dtype=torch.long),
            "loss_mask": torch.tensor(loss_mask, dtype=torch.float),
            "id": datapoint["id"]
        }

In [ ]:
def collate_fn(batch):
    input_ids_lst = [item["input_ids"] for item in batch]
    loss_mask_lst = [item["loss_mask"] for item in batch]
    ids_lst = [item["id"] for item in batch]

    padded_input_ids = pad_sequence(
        input_ids_lst,
        batch_first=True,
        padding_value=token_mapping["<PAD>"]
    )

    padded_loss_mask = pad_sequence(
        loss_mask_lst,
        batch_first=True,
        padding_value=0.0
    )

    return {
        "input_ids": padded_input_ids,
        "loss_mask": padded_loss_mask,
        "ids": ids_lst
    }

In [ ]:
total = config["train_samples"] + config["val_samples"] + config["test_samples"]
full_dataset = StringReversalDataset(min_len=config["min_str_len"], max_len=config["max_str_len"], num_samples=total)

train_dataset, val_dataset, test_dataset = random_split(full_dataset, [config["train_samples"], config["val_samples"], config["test_samples"]])

train_loader = DataLoader(train_dataset, batch_size=config["batch_size"], shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=config["batch_size"], shuffle=False, collate_fn=collate_fn)

In [ ]:
class PositionalEncoding(torch.nn.Module):
    def __init__(self, d_model, max_len, dropout=0.1):
        super().__init__()
        self.dropout = torch.nn.Dropout(p=dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # shape: (1, max_len, d_model) for batch_first

        self.register_buffer("pe", pe)

    def forward(self, x):
        # x shape: (batch, seq_len, d_model)
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


In [ ]:
class DecoderOnlyTransformer(torch.nn.Module):
    def __init__(self, vocab_size, d_model, nhead, num_layers, dim_feedforward, max_seq_len, dropout):
        super().__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.pos_encoder = PositionalEncoding(d_model, max_seq_len, dropout)
        encoder_layer = nn.TransformerEncoderLayer(d_model, nhead, dim_feedforward, dropout, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers)
        self.fc_out = nn.Linear(d_model, vocab_size)

    def forward(self, src, src_pad_mask=None):
        seq_len = src.size(1)
        device = src.device

        # Embed tokens and scale by sqrt(d_model)
        x = self.embedding(src) * math.sqrt(self.d_model)

        # Add positional encoding
        x = self.pos_encoder(x)

        # Generate causal mask
        causal_mask = torch.nn.Transformer.generate_square_subsequent_mask(seq_len).to(device)

        # Pass through transformer
        output = self.transformer(
            x,
            mask=causal_mask,
            src_key_padding_mask=src_pad_mask
        )

        # Get the logits
        logits = self.fc_out(output)

        return logits


In [ ]:
def train_step(engine, batch):
    model.train()
    sequences = batch["input_ids"].to(device)
    loss_masks = batch["loss_mask"].to(device)

    input_seq = sequences[:, :-1]
    target_seq = sequences[:, 1:]
    mask = loss_masks[:, 1:]

    pad_mask = (input_seq == token_mapping["<PAD>"])
    logits = model(input_seq, src_pad_mask=pad_mask)

    loss = torch.nn.functional.cross_entropy(
        logits.reshape(-1, vocab_size), target_seq.reshape(-1), reduction="none"
    )
    loss = (loss * mask.reshape(-1)).sum() / mask.sum()

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    return loss.item()


In [ ]:
def val_step(engine, batch):
    model.eval()

    with torch.no_grad():
        sequences = batch["input_ids"].to(device)
        loss_masks = batch["loss_mask"].to(device)

        input_seq  = sequences[:, :-1]
        target_seq = sequences[:, 1:]
        mask       = loss_masks[:, 1:]

        pad_mask = (input_seq == token_mapping["<PAD>"])
        logits = model(input_seq, src_pad_mask=pad_mask)

        loss = torch.nn.functional.cross_entropy(
            logits.reshape(-1, vocab_size),
            target_seq.reshape(-1),
            reduction='none'
        )
        loss = (loss * mask.reshape(-1)).sum() / mask.sum()

        return loss.item()


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = DecoderOnlyTransformer(
    vocab_size=vocab_size,
    d_model=config["d_model"],
    nhead=config["nhead"],
    num_layers=config["num_layers"],
    dim_feedforward=config["dim_feedforward"],
    max_seq_len=config["max_seq_len"],
    dropout=config["dropout"],
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=config["lr"])

print(f"Device: {device}")

In [ ]:
trainer = Engine(train_step)
evaluator = Engine(val_step)

ProgressBar().attach(trainer, output_transform=lambda x: {"loss": x})

RunningAverage(output_transform=lambda x: x).attach(trainer, "avg_loss")

@trainer.on(Events.EPOCH_COMPLETED)
def run_validation(engine):
    evaluator.run(val_loader)
    val_loss = evaluator.state.output
    train_loss = engine.state.metrics["avg_loss"]
    print(f"Epoch {engine.state.epoch} — Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")


def score_function(engine):
    # returning negative because EarlyStopping looks for maximum
    return -engine.state.output

es_handler = EarlyStopping(patience=10, score_function=score_function, trainer=trainer)
evaluator.add_event_handler(Events.COMPLETED, es_handler)

checkpoint_handler = ModelCheckpoint(
    dirname="checkpoints",
    filename_prefix="best",
    n_saved=1,
    score_function=score_function,
    score_name="neg_val_loss",
    require_empty=False,
)
evaluator.add_event_handler(
    Events.COMPLETED, checkpoint_handler, {"model": model}
)


In [ ]:
def predict(model, input_string):
    model.eval()

    tokens = encode(input_string) + [token_mapping["<SEP>"]]

    with torch.no_grad():
        for _ in range(len(input_string) + 1):
            input_tensor = torch.tensor([tokens], dtype=torch.long).to(device)
            logits = model(input_tensor)
            next_token = logits[0, -1, :].argmax().item()

            if next_token == token_mapping["<EOS>"]:
                break
            tokens.append(next_token)

    sep_idx = len(encode(input_string))
    output_tokens = tokens[sep_idx + 1:]
    return decode(output_tokens)

In [ ]:
def compute_sequence_accuracy(model, dataset):
    model.eval()
    correct = 0

    with torch.no_grad():
        for idx in range(len(dataset)):
            # Subset indexes first need to be mapped to original and then we can access data.
            if hasattr(dataset, 'dataset'):
                real_idx = dataset.indices[idx]
                original = dataset.dataset.all_samples[real_idx]
            else:
                original = dataset.all_samples[idx]

            input_str = original["input"]
            expected = original["output"]
            predicted = predict(model, input_str)

            if predicted == expected:
                correct += 1

    return correct / len(dataset)


In [ ]:
trainer.run(train_loader, max_epochs=config["epochs"])

In [ ]:
print("Checking on some examples (In-distribution)")

test_cases = ["abcdef", "hello", "xyz", "python", "ignite", "gsoc"]
for s in test_cases:
    result = predict(model, s)
    expected = s[::-1]
    status = "Correct!" if result == expected else "Incorrect!"
    print(f"  {status}  {s} → {result} (expected: {expected})")


In [ ]:
val_accuracy = compute_sequence_accuracy(model, val_dataset)
print(f"\nExact-match accuracy (val set): {val_accuracy:.1%}")

test_accuracy = compute_sequence_accuracy(model, test_dataset)
print(f"\nExact-match accuracy (test set): {test_accuracy:.1%}")


In [ ]:
print("Checking on some examples (out of distribution)")

for length in [12, 15, 18, 20]:
    test_str = "".join(random.choices(string.ascii_lowercase, k=length))
    result = predict(model, test_str)
    expected = test_str[::-1]
    status = "Correct!" if result == expected else "Incorrect!"
    print(f"  {status}  len={length}: {test_str} → {result}")
